# Zonal means

Basin zonal means (latitude-depth) of the annual-mean HAMOCC z-level fields, shown as model | observations | model - observations (test | control | test - control when the recipe has an `experiment_cntl` control case, which then replaces the observations).

The model is interpolated to a 1x1 degree grid, the basin mask is applied and the field is averaged over longitude. Observations are put on the same grid and levels, and only cells with both model and observations enter the means.

In [ ]:
import matplotlib.pyplot as plt
import yaml
from pyprojroot import here

from blomster import grid
from hamocc import climatology, compare, io, obs, plots, variables, zonal

In [ ]:
basins = list(variables.BASINS)
compare_obs = True
compare_cntl = True  # use the control case (recipe `experiment_cntl`) when there is one

In [ ]:
with open(here('notebooks_hamocc/settings.yml')) as f:
    cfg = yaml.safe_load(f)
exp = cfg['experiment']
dsets = cfg['datasets']
zcfg = dsets['zonal']

lat, lon, _, _, _, _ = grid.load(cfg['grids'][exp['grid']]['grid_file'])
regridder = zonal.make_regridder(lat, lon)
target = zonal.target_grid()

masks = {b: zonal.load_basin_mask(f"{zcfg['basin_mask_root']}/{zcfg['basin_mask_file'].format(basin=b)}",
                                   zcfg['basin_mask_var']) for b in basins}

names = list(variables.LEVEL_VARS)
needed = sorted({h for n in names for h in climatology.history_names(n)})
cntl = compare.control_case(cfg)
kind = compare.reference_kind(cfg, compare_obs, compare_cntl)

def model_fields(case):
    """Annual-mean fields of a case on the 1x1 grid, one variable at a time."""
    files = io.history_files(case['input_root'], case['caseid'], 'hbgcy', case['year_start'], case['year_end'])
    ds = io.open_history(files, needed)
    fields = {}
    for name in names:
        field = climatology.annual_mean_map(ds, name).load()
        out = regridder(field, skipna=True, na_thres=1.0)
        out.attrs.update(field.attrs)
        fields[name] = out
    return fields

model_1x1 = model_fields(exp)

# reference on the same grid and model depth levels: the control case or the observations
ref_1x1 = {}
if kind == 'control':
    ref_1x1 = model_fields(cntl)
elif kind == 'obs':
    for name in names:
        spec = dsets['maps'].get(name)
        if spec is not None:
            ref_1x1[name] = zonal.on_target_grid(obs.load_obs(spec, dsets['obs_root']), target,
                                                 depth=model_1x1[name]['depth'])

def show(name, basin):
    model = model_1x1[name]
    if name in ref_1x1:
        zm, zr = zonal.basin_zonal_means(model, ref_1x1[name], masks[basin])
    else:
        zm, zr = zonal.zonal_mean(model, masks[basin]), None
    zm.attrs.update(model.attrs)
    title = f"{model.attrs['long_name']}, {basin}"
    if kind == 'control':
        plots.zonal_panels(zm, title=title, zm_control=zr)
    else:
        plots.zonal_panels(zm, zr, title=title)
    plt.show()

```{note}
The basin mask file name and variable (`datasets.zonal` in `recipe_hamocc.yml`) and the observation scale factors (`datasets.units_verified: false`) are not yet verified against the files.
```

## Global

In [ ]:
for name in variables.LEVEL_VARS:
    show(name, 'glb')

## Pacific

In [ ]:
for name in variables.LEVEL_VARS:
    show(name, 'pac')

## Atlantic

In [ ]:
for name in variables.LEVEL_VARS:
    show(name, 'atl')

## Indian

In [ ]:
for name in variables.LEVEL_VARS:
    show(name, 'ind')

## Southern Ocean

In [ ]:
for name in variables.LEVEL_VARS:
    show(name, 'so')